In [2]:
import xarray as xr
import os
import geopandas as gpd
import numpy as np
import xarray as xr
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from multiprocessing import Pool

In [ ]:
# load all files
os.chdir("/Users/anorawu/Team MG Dropbox/Wanru Wu/Cloudseeding/data/SSF")

path = os.getcwd() 
  
# Get the list of all files and directories 
all_files = os.listdir(path) 
data_files_terra = [file for file in all_files if file.find("CERES_SSF_Terra-XTRK_Edition4A")!=-1]

# Load the county-level district file
districts = gpd.read_file(os.path.dirname(os.getcwd()) + "/district/district.shp")
districts = districts.to_crs("EPSG:4326")

# The files were in netCDF3zip format, after manually unzipping, they are netCDF3 now
for file in data_files_terra:
    with xr.open_dataset(data_files_terra[0]) as ds:
        lons = ds["lon"].values
        lats = ds["lat"].values

        # Create GeoDataFrame of points
        point_gdf = gpd.GeoDataFrame(
            geometry=gpd.points_from_xy(lons, lats),
            crs="EPSG:4326"
        )

        # Spatial join
        joined = gpd.sjoin(point_gdf, districts, how="left", predicate="within")

        # Extract date
        dates = ds["time"].values.astype("datetime64[D]").astype("U10")

        # Add back to the netcdf
        ds["region"] = xr.DataArray(joined["dt_adcode"].fillna("None").to_numpy(), dims="time")
        ds["date"] = xr.DataArray(dates, dims="time")
        




3757459


In [63]:
a = pd.to_datetime(ds["time"])
type(a)

pandas.core.indexes.datetimes.DatetimeIndex

In [50]:
ds2 = xr.Dataset(
    {
        "temperature_1": (
            ("lat", "lon"),
            20 * np.random.rand(4).reshape(2, 2),
        ),
        "temperature_2": (("lat", "lon"), 20 * np.random.rand(4).reshape(2, 2)),
    },
    coords={"lat": [10, 20], "lon": [150, 160]},
)
print(ds2)
l = xr.DataArray(np.array([1.7, 2, 3, 4]).reshape(2, 2),dims=['lat','lon'])
ds2["temperature_3"] = l
ds2 = ds2.assign(temperature_4=l)
print(ds2)
print(ds2.isel(lat=[1], lon=[0]))

<xarray.Dataset>
Dimensions:        (lat: 2, lon: 2)
Coordinates:
  * lat            (lat) int64 10 20
  * lon            (lon) int64 150 160
Data variables:
    temperature_1  (lat, lon) float64 11.11 17.06 12.29 17.28
    temperature_2  (lat, lon) float64 14.09 13.05 14.94 9.312
<xarray.Dataset>
Dimensions:        (lat: 2, lon: 2)
Coordinates:
  * lat            (lat) int64 10 20
  * lon            (lon) int64 150 160
Data variables:
    temperature_1  (lat, lon) float64 11.11 17.06 12.29 17.28
    temperature_2  (lat, lon) float64 14.09 13.05 14.94 9.312
    temperature_3  (lat, lon) float64 1.7 2.0 3.0 4.0
    temperature_4  (lat, lon) float64 1.7 2.0 3.0 4.0
<xarray.Dataset>
Dimensions:        (lat: 1, lon: 1)
Coordinates:
  * lat            (lat) int64 20
  * lon            (lon) int64 150
Data variables:
    temperature_1  (lat, lon) float64 12.29
    temperature_2  (lat, lon) float64 14.94
    temperature_3  (lat, lon) float64 3.0
    temperature_4  (lat, lon) float64 3.0
